# This notebook shows you how to use the generate_front_training_data script


In [1]:
import subprocess

## Example of loading 1 sample a week for 4 weeks

In [ ]:
# First you must run

# $env:AWS_ACCESS_KEY_ID="..."
# $env:AWS_SECRET_ACCESS_KEY="..."
#
# or
#
# export AWS_ACCESS_KEY_ID=...
# export AWS_SECRET_ACCESS_KEY=...


In [3]:
# 4 samples, 1 from each of 4 consecutive weeks

# temporal options ------
timestep_hours = "672"        # how many hours to load total. This example is 4 weeks
sampling_step = "168"         # stride in hours, 1 week
start_record = "1180"         # first valid wind/forcing record begins ~1180, This is also the default start of the script

# spatial options ------
target_km_res = "150"         # km resolution of our patches w and h
down_sample_res = "64"        # pixel resolution of our patches w and h
# NOTE The above are heavily dependent on each other. down_sample_res must be smaller than the smallest possible resolution created by target_km_re on the llc grid. 150 and 64 are safe. Todo it would be nice to create a function to calculate this for the user.
samples_per_snapshot = "25"   # how many patches to take from each snapshot
bias_to_high_gradients = "2"  # see documentation on sampling

# s3 output config ------
# Note: include trailing "/" please
bucket = "llc/"
folder = "native_grid_dbof_training_data/"
run_id = "script_test_00/"   # should be unique for each instance of dbof you wish to create, it is just a folder in s3

s3_endpoint = "https://s3-west.nrp-nautilus.io"

# parallel ------
num_workers = 20 # How many threads would you like to create? Note : Should be <= sample_points_per_snapshot

In [4]:
# It takes a while to build the script dependencies and setup
print()
!generate-llc-dataset --sampling_step {sampling_step} --start_record {start_record} --timestep_hours {timestep_hours} --sample_points_per_snapshot 25 --bias_to_high_gradients 2 --target_km_res 150 --down_sample_res 64 --bucket llc/ --folder native_grid_dbof_training_data/ --run_id script_test_00/ --s3_endpoint https://s3-west.nrp-nautilus.io --num_workers 20



^C


In [ ]:

cmd = [
    "generate-llc-dataset",

    # Sampling
    "--sampling_step", str(sampling_step),
    "--start_record", str(start_record),
    "--timestep_hours", str(timestep_hours),
    "--sample_points_per_snapshot", str(samples_per_snapshot),
    "--bias_to_high_gradients", str(bias_to_high_gradients),
    "--target_km_res", str(target_km_res),
    "--down_sample_res", str(down_sample_res),

    # Output / storage
    "--bucket", bucket,
    "--folder", folder,
    "--run_id", run_id,
    "--s3_endpoint", s3_endpoint,

    # Parallelism
    "--num_workers", str(num_workers),
]
subprocess.run(cmd, shell=True, capture_output=True)

In [ ]:
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://llc/native_grid_dbof_training_data/script_test_00/ --human-readable
#
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://llc/native_grid_dbof_training_data/script_test_00/ --recursive --dryrun